In [2]:
!pip install --upgrade pip setuptools wheel
!pip install implicit
!pip install pandas
!pip install sklearn
!pip install pyarrow
!pip install catboost

  Using cached pip-22.3-py3-none-any.whl (2.1 MB)
  Attempting uninstall: pip
    Found existing installation: pip 22.2.2
    Uninstalling pip-22.2.2:
      Successfully uninstalled pip-22.2.2


In [1]:
import tqdm
import numpy as np
import pandas as pd
from tqdm import tqdm

import gc
import implicit_features
import utils

#import pop_candidates

# df_train = pd.read_parquet('data/train.par')#.astype({"ts": "datetime64[ms]"})
# df_train['t_dat'] = df_train['ts']
# df_train = df_train.rename({'session': 'customer_id', 'aid': 'article_id'}, axis=1).astype({"t_dat": "datetime64[ms]"})
type_labels = {'clicks':0, 'carts':1, 'orders':2}
df_test = pd.read_parquet('data/test.par')#.astype({"ts": "datetime64[ms]"})
df_test['t_dat'] = df_test['ts']

df_test = df_test.rename({'session': 'customer_id', 'aid': 'article_id'}, axis=1).astype({"t_dat": "datetime64[ms]"})
df_test['type'] = df_test['type'].map(type_labels)

In [3]:
df_train = pl.read_parquet('data/split_sessions.par')

In [1]:
import polars as pl

In [3]:
df = pl.read_parquet('data/df_predicted_all_lgbm.par')

In [4]:
df.write_parquet('data/df_lb_predict_all.par')

In [1]:
import polars as pl
import tqdm
import numpy as np
import pandas as pd
from tqdm import tqdm

import gc
import implicit_features
import utils

#df_train = pl.read_parquet('data/train.par')#.astype({"ts": "datetime64[ms]"})
#df_train = df_train.rename({'session': 'customer_id', 'aid': 'article_id'})
df_test = pl.read_parquet('data/test.par')#.astype({"ts": "datetime64[ms]"})
df_train = pl.read_parquet('data/split_sessions.par')
df_train = df_train.with_column(
       pl.from_epoch("ts", unit="ms").alias('t_dat')
   )

df_test = df_test.with_column(
       pl.from_epoch("ts", unit="ms").alias('t_dat')
   )

type_labels = {'clicks': 0, 'carts': 1, 'orders': 2}

type_labels = pl.DataFrame({
    "type": list(type_labels.keys()),
    "values": list(type_labels.values())
})
df_train = df_train.join(type_labels, on="type", how="left").drop(columns=['type']).rename({'values': 'type'})
df_test = df_test.join(type_labels, on="type", how="left").drop(columns=['type']).rename({'values': 'type'})


df_test = df_test.rename({'session': 'customer_id', 'aid': 'article_id'})
test_users = df_test['customer_id'].unique()
df = pl.concat([df_train.select(df_test.columns), df_test])

df = df.sort('t_dat', reverse=True)
df = df.with_column(pl.col("ts").
               rank(method="ordinal", reverse=True).
               over(["customer_id"]).alias("num_action"))

def split_data(df, validation_cut, test_users):
    df_val = df.filter((pl.col("num_action") <= validation_cut) & (pl.col('customer_id').is_in(test_users)))
    df_train = df.filter((pl.col("num_action") > validation_cut) | (~pl.col('customer_id').is_in(test_users)))
    return df_train, df_val

PREDICT_LB = True
if PREDICT_LB:
    print('PREDICT ON LB')
    df_train_all = df
    df_train_traks, df_test_traks = split_data(df_train_all, validation_cut=3, test_users=test_users)
    df_train_traks = df_train_traks.sort('t_dat', reverse=True)
    df_train_traks = df_train_traks.with_column(pl.col("ts").
                rank(method="ordinal", reverse=True).
                over(["customer_id"]).alias("num_action"))
 #   df_train_traks['num_action'] = df_train_traks['num_action'] - 2
    
else:
    # LOCAL VALIDATION
    df_train_traks, df_test_traks = split_data(df, validation_cut=1, test_users=test_users)
    df_train_traks = df_train_traks.sort_values('t_dat', ascending=False)
    df_train_traks['num_action'] = df_train_traks.groupby('customer_id')['article_id'].cumcount()

del df_train
del df
# df_train_traks = df_train_traks.to_pandas()
# df_test_traks = df_test_traks.to_pandas()
# df_train_all = df_train_all.to_pandas()       

PREDICT ON LB


df_train = df_train.sort_values('t_dat', ascending=True)
df_train['shift_time'] = df_train.groupby('customer_id')['t_dat'].shift()
df_train['session'] = np.where(df_train['t_dat'] - df_train['shift_time']>pd.Timedelta(hours=1), 1, 0)
df_train['session'] = df_train.groupby('customer_id')['session'].cumsum()
df_train['max_date'] = df_train['customer_id'].map(df_train.groupby(['customer_id'])['t_dat'].max())
df_train['split_1'] = np.where((df_train['t_dat'] - df_train['shift_time']>pd.Timedelta(hours=1)) & (df_train['max_date'] - df_train['t_dat'] < pd.Timedelta(days=7)), 1, 0)
df_train['split_1'] = df_train.groupby(['customer_id'])['split_1'].cumsum()
df_train['split_1'] = np.where(df_train['split_1']>0, 1, 0)
df_train['split_2'] = np.where((df_train['t_dat'] - df_train['shift_time']>pd.Timedelta(hours=1)) & (df_train['max_date'] - df_train['t_dat'] < pd.Timedelta(days=14)),
         1, 0)
df_train['split_2'] = df_train.groupby(['customer_id'])['split_2'].cumsum()
df_train['split_2'] = np.where(df_train['split_2']>0, 1, 0)
df_train['split_2'] = np.where(df_train['split_1']==1, 0, df_train['split_2'] )
df_train['split_3'] = np.where((df_train['split_1']==1) | (df_train['split_2']==1), 0, 1 )
tmp = df_train[df_train['split_1']==1]
tmp['customer_id'] = -tmp['customer_id'] 
df_train = pd.concat([df_train, tmp])
tmp = df_train[df_train['split_2']==1]
import gc
gc.collect()
tmp['customer_id'] = -tmp['customer_id'] + df_train['customer_id'].min()
df_train = pd.concat([df_train, tmp])
tmp = df_train[df_train['split_3']==1]
tmp['customer_id'] = -tmp['customer_id'] + df_train['customer_id'].min()
df_train = pd.concat([df_train, tmp])
df_train = df_train[df_test.columns]#.head()
count_user = df_train['customer_id'].value_counts()
df_train = df_train[df_train['customer_id'].isin(count_user[count_user>1].keys())]

# df_test_traks = df_test_traks.to_pandas()
# type_labels = {0: 'clicks', 1: 'carts', 2: 'orders'}
# df_test_traks['type'] = df_test_traks['type'].map(type_labels)
#df_train = df.to_pandas().sort_values('joint_rnk_5')
submission = utils.sub_for_metrics(w2v_rec, df_train_traks.to_pandas(), list(df_test_traks['customer_id'].unique()))
utils.get_metric(submission, df_test_traks)

In [8]:
df_train.to_parquet('data/split_sessions.par')

In [3]:
import public_candidates
public_candidates_train = public_candidates.predict(df_train_traks[~df_train_traks['customer_id'].isin(test_users)], 
                                                    df_train_traks[df_train_traks['customer_id'].isin(test_users)])
public_candidates_train.to_parquet('data/public_cand_lb_train.par')

  0%|          | 0/13054 [00:00<?, ?it/s]

  0%|          | 0/13054 [00:03<?, ?it/s]

  0%|          | 0/456 [00:01<?, ?it/s]

In [2]:
public_candidates_train = pd.read_parquet('data/public_cand_lb_train.par')

train_user = df_train['customer_id'].unique()[:int(df_train['customer_id'].nunique()*0.5)]
df_train = df_train[df_train['customer_id'].isin(train_user)]

import random
session_sample = random.sample(list(df['customer_id'].unique()), df['customer_id'].nunique()//6)

df = df[df['customer_id'].isin(session_sample)]
test_users = list(set(test_users).intersection(set(session_sample)))

In [2]:
df_als_train = implicit_features.als(df_train_traks, test_users, name_model='als')

  0%|          | 0/30 [00:00<?, ?it/s]

In [5]:
pred_users = set(test_users)
print(len(pred_users))
df_train = implicit_features.make_pred(df_train_traks.select(['customer_id', 'article_id', 'num_action', 't_dat']).to_pandas(), 
                                      list(pred_users), filter_already_liked_items=False)
if PREDICT_LB:
    df_train.to_parquet('data/implicit_lb_train.par')
else:
    df_train.to_parquet('data/implicit_val_train.par')

1671803


  0%|          | 0/1855536 [00:00<?, ?it/s]

In [2]:
df_predicted = implicit_features.make_pred(df_train_all.select(['customer_id', 'article_id', 'num_action', 't_dat']).to_pandas(), 
    test_users, filter_already_liked_items=False)
df_predicted.to_parquet('data/implicit_lb_predict.par')

  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [05:15<00:00, 5304.47it/s]


  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [03:43<00:00, 7491.60it/s]


  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [06:00<00:00, 4634.45it/s]


  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [06:00<00:00, 4632.04it/s]


  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [05:22<00:00, 5188.65it/s]


In [3]:
df_train_10 = implicit_features.make_pred(df_train_traks.select(['customer_id', 'article_id', 'num_action', 't_dat']).to_pandas(), 
                                    df_test_traks['customer_id'].unique(), train_day=11)
for col in set(df_train_10.columns) - set(['article_id', 'customer_id']):
    df_train_10 = df_train_10.rename(columns={col: col+'_10'})
    
df_train_10.to_parquet('data/implicit_lb_train10.par')

df_predicted_10 = implicit_features.make_pred(df_train_all.select(['customer_id', 'article_id', 'num_action', 't_dat']).to_pandas(), 
                                    test_users, train_day=11)
for col in set(df_predicted_10.columns) - set(['article_id', 'customer_id']):
    df_predicted_10 = df_predicted_10.rename(columns={col: col+'_10'})
    
df_predicted_10.to_parquet('data/implicit_lb_predict10.par')




pred_users = set(test_users)
print(len(pred_users))
df_train_orders_carts = implicit_features.make_pred(df_train_traks[df_train_traks['type']!='clicks'], 
                                    list(pred_users), filter_already_liked_items=False)
                                    
for col in (set(df_train_orders_carts.columns) - set(['article_id', 'customer_id'])):
    df_train_orders_carts = df_train_orders_carts.rename({col: col+'_order'}, axis=1)

if PREDICT_LB:
    df_train_orders_carts.to_parquet('data/implicit_lb_train_orders.par')
else:
    df_train.to_parquet('data/implicit_val_train.par')

In [6]:
df_train = pl.read_parquet('data/implicit_lb_train.par', columns=['article_id', 'customer_id'])
#df_train = pd.merge(df_train, pd.read_parquet('data/implicit_lb_train_orders.par'), on=['article_id', 'customer_id'], how='outer')
df_train_10 = pl.read_parquet('data/implicit_lb_train10.par', columns=['article_id', 'customer_id'])

df_train = df_train.join(df_train_10, on=['customer_id', 'article_id'], how='outer')
del df_train_10

(85681647, 2)
(14884055, 3)
shape: (2, 2)
┌────────┬──────────┐
│ target ┆ count    │
│ ---    ┆ ---      │
│ i32    ┆ u32      │
╞════════╪══════════╡
│ 0      ┆ 14753351 │
├╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌┤
│ 1      ┆ 130704   │
└────────┴──────────┘


In [39]:
def add_last(df, df_track, pred_users):
    tmp = df_track.filter(pl.col('customer_id').is_in(pred_users))
    tmp = tmp.groupby(["customer_id","article_id"]).agg([pl.col('type').count().alias('count'),
         pl.col('type').mean().alias('mean'), ]).sort(['mean', 'count'], reverse=True)
    df = df.join(tmp.groupby(['customer_id']).head(50).select(['customer_id', 'article_id']), on=['customer_id', 'article_id'], how='outer')
    return df

In [35]:
def make_candidates(df_traks, mode='train', df_test_traks=None, pred_users=None):
    df_train = pl.read_parquet(f'data/implicit_lb_{mode}.par', columns=['article_id', 'customer_id'])
    #df_train = pd.merge(df_train, pd.read_parquet('data/implicit_lb_train_orders.par'), on=['article_id', 'customer_id'], how='outer')
    df_train_10 = pl.read_parquet(f'data/implicit_lb_{mode}10.par', columns=['article_id', 'customer_id'])
    df_train = df_train.join(df_train_10, on=['customer_id', 'article_id'], how='outer')
    print(df_train.shape)
   # df_train = add_last(df_train, df_traks, pred_users)
    joint_traks = pl.read_parquet(f'data/{mode}_coo_matrix.par')
    df_train = agg_ratio_joint_sales(df_train, joint_traks, df_traks, pred_users, N=1, for_candidate=True )
   # df_train = agg_ratio_joint_sales(df_train, joint_traks, df_traks, pred_users, N=3, for_candidate=True )
   # df_train = df_train.join(df_joint, how='outer', on=['customer_id', 'article_id'])
    print(df_train.shape, 'add_last')
    if mode=='train':
        df_test_traks = df_test_traks.with_column(pl.lit(1).alias("target"))
        df_train_candidat = df_train.join(df_test_traks.filter(pl.col('type')!=0).select(['target', 'customer_id', 'article_id']), how='left', on=['customer_id', 'article_id'])
        df_train_candidat = df_train_candidat.with_column(pl.col("target").fill_null(pl.lit(0)) )
        df_train_candidat = df_train_candidat.join(df_train_candidat.groupby(['customer_id']).agg(pl.col('target').max().alias('max_target')), on=['customer_id'])
        df_train_candidat = df_train_candidat.filter(pl.col('max_target')==1).drop(columns=['max_target'])
        print(df_train_candidat.groupby(['target']).count(), 'all_candidates')
        neg_sample = len(df_train_candidat.filter(pl.col('target')==1))*20
        df_train_candidat = pl.concat([df_train_candidat.filter(pl.col('target')==1), df_train_candidat.filter(pl.col('target')==0).sample(neg_sample)])
        print(df_train_candidat.groupby(['target']).count())
        df_train_candidat.write_parquet(f'data/df_{mode}_candidat.par')
    elif mode=='predict':
        df_train.write_parquet(f'data/df_{mode}_candidat.par')


In [38]:
def agg_ratio_joint_sales(df, joint_items, df_traks, pred_users, N=5, for_candidate=False ):
    df_traks_tmp = df_traks.filter((pl.col('num_action')<=N)& (pl.col('customer_id').is_in(pred_users))).select(['customer_id', 'article_id'])
    df_traks_tmp = df_traks_tmp.join(joint_items.filter(pl.col('rnk')<=15), on='article_id') 
    df_traks_tmp = df_traks_tmp.groupby(['customer_id', 'article_id_right']).agg(pl.col('score').sum().alias(f'joint_score_{N}'))

    if for_candidate:
        df_traks_tmp = df_traks_tmp.with_column(pl.col(f'joint_score_{N}').
                rank(method="dense", reverse=True).
                over(["customer_id"]).alias(f'rnk'))
        df = df.join(df_traks_tmp.filter(pl.col('rnk')<=10).rename({'article_id_right': 'article_id'}), how='outer', on=['customer_id', 'article_id'])

        return df
    else:
        df = df.join(df_traks_tmp.rename({'article_id_right': 'article_id'}), how='left', on=['customer_id', 'article_id'])
        df = df.with_column(pl.col(f'joint_score_{N}').
                    rank(method="dense", reverse=True).
                    over(["customer_id"]).alias(f'joint_rnk_{N}'))

        return df

In [39]:
make_candidates(df_train_traks, mode='train', df_test_traks=df_test_traks, pred_users=df_test_traks['customer_id'].unique())
make_candidates(df_train_all, mode='predict')

(64265449, 2)
(66238162, 4) add_last
shape: (2, 2)
┌────────┬──────────┐
│ target ┆ count    │
│ ---    ┆ ---      │
│ i32    ┆ u32      │
╞════════╪══════════╡
│ 0      ┆ 14098127 │
├╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌┤
│ 1      ┆ 130452   │
└────────┴──────────┘ all_candidates
shape: (2, 2)
┌────────┬─────────┐
│ target ┆ count   │
│ ---    ┆ ---     │
│ i32    ┆ u32     │
╞════════╪═════════╡
│ 0      ┆ 2609040 │
├╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌┤
│ 1      ┆ 130452  │
└────────┴─────────┘
(223552921, 2)
(223552921, 4) add_last


In [31]:
129988 / 13866969

0.009373930236665273

In [28]:
130452 / 14098127

0.009253144052397883

In [46]:
129487 / 13583335

0.009532784106406858

In [47]:
129498 / 13587946

0.009530358745906114

In [ ]:
129492
129487 - bez

In [ ]:
64290565
64265449

In [4]:
130704 / 14753351

0.00885927542834167

df_train = pd.merge(df_train, public_candidates_train, on=['customer_id', 'article_id'], how='outer')
del public_candidates_train

for col in set(df_train.columns) - set(['article_id', 'customer_id']):
  df_train[col] = df_train[col].astype(np.float32)
df_train['article_id'] = df_train['article_id'].astype(np.int32)

df_train_5 = pd.read_parquet('data/train_5_boosting.par', columns=['rnk_tifuknn_tfidf_5', 'rnk_BM25_5', 'rnk_tifu_cos_5', 
    'rnk_tfidf_5', 'rnk_cos412_5', 'customer_id', 'article_id'])

print(df_train.shape)
df_train = pd.merge(df_train, df_train_5, on=['customer_id', 'article_id'], how='outer')
print(df_train.shape)


del df_train_5
gc.collect()

df_train_elsa = pd.read_parquet('data/df_predicted_train_elsa.par')
df_train_elsa = df_train_elsa.sort_values('pred', ascending=False)
df_train_elsa['rnk_elsa'] = df_train_elsa.groupby('customer_id')['pred'].cumcount()
df_train_elsa = df_train_elsa[df_train_elsa['rnk_elsa']<85]
df_train = pd.merge(df_train, df_train_elsa[['customer_id', 'article_id', 'pred']], on=['customer_id', 'article_id'], how='outer')

df_train_als = pd.read_parquet('data/df_train_als.par')
df_train_als = df_train_als.sort_values('score_als', ascending=False)
df_train_als['rnk_als'] = df_train_als.groupby('customer_id')['score_als'].cumcount()
df_train_als = df_train_als[df_train_als['rnk_als']<70]
df_train = pd.merge(df_train, df_train_als, on=['customer_id', 'article_id'], how='outer')

df_train_als = implicit_features.als(df_train_traks, df_test_traks['customer_id'].unique())

df_train_als.to_parquet('data/df_train_als.par')
df_train_als = pd.read_parquet('data/df_train_als.par')
df_train = pd.merge(df_train, df_train_als, on=['customer_id', 'article_id'], how='outer')

In [40]:
def create_cooc_stats(df, min_rank_diff: int = -4, max_rank_diff: int = 4, items_per_item: int = 10):
   # df = pl.DataFrame(df)
    batch = 350000
    df = df.filter(pl.col('num_action')<=50)
    users = df['customer_id'].unique().shuffle(seed=1)
    result = []
    for i in tqdm(range(len(users)//batch+1)):
        df_train_traks_tmp = df.filter(pl.col('customer_id').is_in(users[i*batch: i*batch+batch])).select(['article_id', 'customer_id', 'num_action'])
        coocf = df_train_traks_tmp.join(df_train_traks_tmp, on=['customer_id'])#.
        coocf = coocf.filter((pl.col('num_action') - pl.col('num_action_right') >= min_rank_diff) & (pl.col('num_action') - pl.col('num_action_right') <= max_rank_diff))
        coocf = coocf.filter((pl.col('article_id')!=pl.col('article_id_right')))
        coocf = coocf.groupby(['article_id', 'article_id_right']).agg(pl.count())
        result.append(coocf)
        if i%20==0:
            result = [pl.concat(result).groupby(['article_id', 'article_id_right']).agg(pl.col('count').sum())]
        
    groups = pl.concat(result).groupby(['article_id', 'article_id_right']).agg(pl.col('count').sum())
    groups = groups.join(groups.groupby(['article_id']).agg(pl.col('count').sum().alias("count_item1")), on=['article_id'])
    groups = groups.filter(pl.col('count_item1')>=items_per_item)
    groups = groups.with_column((groups['count'] / groups['count_item1']).alias("score"))
    groups = groups.with_column(pl.col("score").
               rank(method="min", reverse=True).
               over(["article_id"]).alias("rnk"))

    return groups

def agg_ratio_joint_sales(df, joint_items, df_traks, pred_users, N=5, for_candidate=False ):
    df_traks_tmp = df_traks.filter((pl.col('num_action')<=N)& (pl.col('customer_id').is_in(pred_users))).select(['customer_id', 'article_id'])
    df_traks_tmp = df_traks_tmp.join(joint_items.filter(pl.col('rnk')<=15), on='article_id') 
    df_traks_tmp = df_traks_tmp.groupby(['customer_id', 'article_id_right']).agg(pl.col('score').sum().alias(f'joint_score_{N}'))

    if for_candidate:
        df_traks_tmp = df_traks_tmp.with_column(pl.col(f'joint_score_{N}').
                rank(method="dense", reverse=True).
                over(["customer_id"]).alias(f'rnk'))
        df = df.join(df_traks_tmp.filter(pl.col('rnk')<=20).rename({'article_id_right': 'article_id'}), how='outer', on=['customer_id', 'article_id'])

        return df
    else:
        df = df.join(df_traks_tmp.rename({'article_id_right': 'article_id'}), how='left', on=['customer_id', 'article_id'])
        df = df.with_column(pl.col(f'joint_score_{N}').
                    rank(method="dense", reverse=True).
                    over(["customer_id"]).alias(f'joint_rnk_{N}'))

        return df

In [3]:
joint_items = create_cooc_stats(df_train_traks.filter(pl.col('customer_id')>=0))

100%|██████████| 39/39 [05:22<00:00,  8.26s/it]


df = agg_ratio_joint_sales(joint_items, joint_items, df_train_traks.filter(pl.col('customer_id')>=0), pred_users=df_test_traks['customer_id'].unique(), N=5, 
for_candidate=True)
df_test_traks = df_test_traks.to_pandas()
type_labels = {0: 'clicks', 1: 'carts', 2: 'orders'}
df_test_traks['type'] = df_test_traks['type'].map(type_labels)


In [4]:
df_rnn = pd.read_csv('data/df_train_rnn.csv')

In [15]:
def to_int(str_):
    try:
        return int(str_)
    except:
        return ' '

In [21]:
df_rnn['labels'] = df_rnn.preds.apply(lambda x: [to_int(i) for i in x[1:-1].split(' ')])

In [ ]:
df_rnn['labels'] = df_rnn.preds.apply(lambda x: [to_int(i) for i in x[1:-1].split(' ')])

In [19]:
df_rnn.labels

0         [ , 582732, 1142000, 1502122, 1758603,  , 7605...
1         [ , 199008, 1200570,  , 811084, 1583317,  , 75...
2         [1007613,  , 409367,  , 383474,  , 670423, 177...
3         [ , 351665, 1216820,  , 189653, 1387003, 11690...
4         [ , 910034, 1579935,  , 935315,  , 988806,  , ...
                                ...                        
466820    [1839213,  ,  , 69440,  , 907693,  , 226839,  ...
466821    [ , 779748,  , 781377,  , 516867, 1718352, 125...
466822    [1052749, 1639897, 1748631, 1204986, 1501705, ...
466823    [1059174,  , 164613, 1128777,  , 279041,  , 93...
466824    [ , 272221,  , 279269, 1152891, 1116621, 10998...
Name: preds, Length: 466825, dtype: object

In [22]:
df_rnn

,session,preds,labels
0,12899780,[ 582732 1142000 1502122 1758603 760500 13656...,"[ , 582732, 1142000, 1502122, 1758603, , 7605..."
1,12899781,[ 199008 1200570 811084 1583317 759436 9507...,"[ , 199008, 1200570, , 811084, 1583317, , 75..."
2,12899782,[1007613 409367 383474 670423 1779730 4101...,"[1007613, , 409367, , 383474, , 670423, 177..."
3,12899783,[ 351665 1216820 189653 1387003 1169057 1871...,"[ , 351665, 1216820, , 189653, 1387003, 11690..."
4,12899784,[ 910034 1579935 935315 988806 707586 9180...,"[ , 910034, 1579935, , 935315, , 988806, , ..."
...,...,...,...
466820,14571511,[1839213 69440 907693 226839 85930 4203...,"[1839213, , , 69440, , 907693, , 226839, ..."
466821,14571515,[ 779748 781377 516867 1718352 1252670 14868...,"[ , 779748, , 781377, , 516867, 1718352, 125..."
466822,14571517,[1052749 1639897 1748631 1204986 1501705 7425...,"[1052749, 1639897, 1748631, 1204986, 1501705, ..."
466823,14571519,[1059174 164613 1128777 279041 938374 1383...,"[1059174, , 164613, 1128777, , 279041, , 93..."


df = df.sort('joint_score_5', reverse=True).rename({'article_id_right': 'article_id'})
df_train = df.to_pandas()#.sort_values('joint_rnk_5')
submission = utils.sub_for_metrics(df_train, df_train_traks.to_pandas(), list(df_test_traks['customer_id'].unique()))
utils.get_metric(submission, df_test_traks)

In [8]:
joint_items.select(['article_id', 'article_id_right', 'score', 'rnk']).write_parquet('data/train_coo_matrix.par')

In [9]:
joint_items = create_cooc_stats(df_train_all.filter(pl.col('customer_id')>=0))
joint_items.select(['article_id', 'article_id_right', 'score', 'rnk']).write_parquet('data/predict_coo_matrix.par')

100%|██████████| 42/42 [05:43<00:00,  8.18s/it]


In [41]:
def time_cat(df, days=7):
    return df.filter(pl.col('ts') >= df['ts'].max() - days*1000*60*60*24)

def make_features(df_traks, mode='metric'#, item_factors, user_factors
):
     # расстояние между действием и покупкой
     # Среднее время между событиями
     # фичи по времени и дню недели как профиль клиента 
     df_traks = df_traks.with_columns(pl.col('t_dat').dt.hour().alias('hour'))
     df_traks = df_traks.with_columns(pl.col('t_dat').dt.weekday().alias('weekday'))
     
     # Средняя неделя и час покупки товара

     if mode=='train':
          df = pl.read_parquet(f'data/df_{mode}_candidat.par', columns=['customer_id', 'article_id', 'target'])
     else:
          df = pl.read_parquet(f'data/df_{mode}_candidat.par', columns=['customer_id', 'article_id'])
    
     last_view = df_traks.groupby(['article_id']).agg(pl.col('ts').max() ).with_columns(((df_traks['ts'].max() - pl.col('ts'))//1000).alias('last_view'))
     df = df.join(last_view.select(['article_id', 'last_view']), how='left', on=['article_id'])

     df_orders = df_traks.filter(pl.col('type')!=0)
     last_view = df_orders.groupby(['article_id']).agg(pl.col('ts').max() ).with_columns(((df_traks['ts'].max() - pl.col('ts'))//1000).alias('last_buy'))
     df = df.join(last_view.select(['article_id', 'last_buy']), how='left', on=['article_id'])
     joint_traks = pl.read_parquet(f'data/{mode}_coo_matrix.par')
    # joint_traks = get_joint_listren(df_traks, 'article_id')
     df = agg_ratio_joint_sales(df, joint_traks, df_traks, pred_users=df['customer_id'].unique(), N=30)
     df = agg_ratio_joint_sales(df, joint_traks, df_traks, pred_users=df['customer_id'].unique(), N=15)
     #return df, joint_traks
     df = agg_ratio_joint_sales(df, joint_traks, df_traks, pred_users=df['customer_id'].unique(), N=5)
     df = agg_ratio_joint_sales(df, joint_traks, df_traks, pred_users=df['customer_id'].unique(), N=1)
     del joint_traks
   # return df
     df_orders = df_orders.with_column(pl.col("ts").
          rank(method="min", reverse=True).
          over(["customer_id"]).alias("num_action"))
     joint_traks = create_cooc_stats(df_orders, min_rank_diff = -3, max_rank_diff = 3, items_per_item=3)

     df = agg_ratio_joint_sales(df, joint_traks, df_orders, pred_users=df['customer_id'].unique(), N=2)
     df = agg_ratio_joint_sales(df, joint_traks, df_orders, pred_users=df['customer_id'].unique(), N=10)

     df_tmp = df_traks.groupby(['customer_id', 'article_id']).agg([pl.col('num_action').min().alias('min_num_action'), 
          pl.col('num_action').count().alias('user_item_count')])
     df = df.join(df_tmp, on=['customer_id', 'article_id'], how='left')
     #return df
     df_tmp = df_orders.groupby(['customer_id', 'article_id']).agg(pl.col('num_action').count().alias('user_item_count_orders'))
     df = df.join(df_tmp, on=['customer_id', 'article_id'], how='left')

     # df = pd.merge(df, joint_traks.rename({'article_id_x': 'article_id', 'article_id_y': 'last_article_id'}, axis=1), how='left', on=['article_id', 'last_article_id'])
     # df = df.drop(columns=['last_article_id'])
     df_implicit = pl.read_parquet(f'data/implicit_lb_{mode}.par').drop(columns=['__index_level_0__']) # predict
     df = df.join(df_implicit, on=['article_id', 'customer_id'], how='left')
     df_implicit = pl.read_parquet(f'data/implicit_lb_{mode}10.par').drop(columns=['__index_level_0__']) # predict
     df = df.join(df_implicit, on=['article_id', 'customer_id'], how='left')

     #return df
     # df_implicit = pd.read_parquet(f'data/implicit_lb_{mode}_orders.par') # predict
     # df = pd.merge(df, df_implicit, on=['article_id', 'customer_id'], how='left')

     gc.collect()


     # Среднии, минимальные и максимальные ранги после модели первого уровня для артистов и треков
     df = df.with_columns(df[['rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].mean(axis=1).alias('mean_rnk_cand'))
     df = df.with_columns(pl.Series(df.select(['rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']).to_pandas().std(axis=1)).alias('std_rnk_cand'))
     df = df.with_columns(df[['rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].min(axis=1).alias('min_rnk_cand'))
     df = df.with_columns(df[['rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].max(axis=1).alias('max_rnk_cand'))

     df = df.with_columns(df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos_10','rnk_tfidf_10',  'rnk_cos412_10', 'rnk_BM25_10',]].mean(axis=1).alias('mean_rnk_cand_10'))
     df = df.with_columns(pl.Series(df.select(['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos_10','rnk_tfidf_10',
           'rnk_cos412_10', 'rnk_BM25_10']).to_pandas().std(axis=1)).alias('std_rnk_cand_10'))
     df = df.with_columns(df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos_10','rnk_tfidf_10',  'rnk_cos412_10', 'rnk_BM25_10']].min(axis=1).alias('min_rnk_cand_10'))
     df = df.with_columns(df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos_10','rnk_tfidf_10',  'rnk_cos412_10', 'rnk_BM25_10']].max(axis=1).alias('max_rnk_cand_10'))

     df = df.with_columns(df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos_10','rnk_tfidf_10',  'rnk_cos412_10', 'rnk_BM25_10',
          'rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].mean(axis=1).alias('mean_rnk_cand_all'))
     df = df.with_columns(pl.Series(df.select(['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos_10','rnk_tfidf_10',  'rnk_cos412_10', 'rnk_BM25_10',
          'rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']).to_pandas().std(axis=1)).alias('std_rnk_cand_all'))

     
     # фичи на популярность пользователя, товара 
     tmp = df_traks.groupby(['article_id']).count().rename({'count':'pop_article_id'})
     df = df.join(tmp, on=['article_id'], how='left')
     # Ранги песен внутри кандидатов
     df = df.with_column(pl.col("pop_article_id").
          rank(method="min", reverse=True).
          over(["customer_id"]).alias("pop_item_rnk_user"))

     df_traks = df_traks.join(tmp, on=['article_id'], how='left')
     df = df.join(df_traks.groupby(['customer_id']).agg(pl.col('article_id').n_unique().alias('nunique_article_id')), on=['customer_id'], how='left')
     
     tmp = df_traks.groupby(['customer_id']).count().rename({'count':'pop_user'})
     df = df.join(tmp, on=['customer_id'], how='left')
     tmp = df_traks.filter(pl.col('type')==1).groupby(['customer_id']).count().rename({'count':'pop_user_carts'})
     df = df.join(tmp, on=['customer_id'], how='left')
     tmp = df_traks.filter(pl.col('type')==2).groupby(['customer_id']).count().rename({'count':'pop_user_orders'})
     df = df.join(tmp, on=['customer_id'], how='left')
     
     
     # Какие треки в среднем по популярности слушает пользователь
     tmp = df_traks.groupby(['customer_id']).agg([pl.col('pop_article_id').std().alias('pop_article_id_std_user'), 
          pl.col('pop_article_id').mean().alias('pop_article_id_mean_user')])
     df = df.join(tmp, on=['customer_id'], how='left')
     df = df.with_columns((pl.col('pop_article_id_mean_user')/ pl.col('pop_article_id')).alias('ratio_pop_user_item'))
     #return df
     for i in [5, 10, 15]:
          tmp_track = time_cat(df_traks, days=i)
         # tmp_track = df_traks[df_traks['num_action']<=i]

          tmp = tmp_track.groupby(['article_id']).count().rename({'count':f'pop_item_{i}'})
          df = df.join(tmp, on=['article_id'], how='left')
          df = df.with_column(pl.col(f'pop_item_{i}').
               rank(method="dense", reverse=True).
               over(["customer_id"]).alias(f"pop_item_rnk_user_{i}"))

          tmp = tmp_track.filter(pl.col('type')==2).groupby(['article_id']).count().rename({'count':f'pop_item_orders_{i}'})
          df = df.join(tmp, on=['article_id'], how='left')
          # Добавить события пользователя

     #return df
     tmp = df_traks.groupby('customer_id').agg([pl.col('ts').min().alias('min_date_session'), 
           pl.col('ts').max().alias('max_date_session')])
     tmp = tmp.with_columns(((pl.col('max_date_session') - pl.col('min_date_session'))//1000).alias('leng_session'))
     df = df.join(tmp.select(['customer_id', 'leng_session']), how='left', on=['customer_id'])
     df = df.with_columns((pl.col('pop_user') / pl.col('leng_session')).alias('items_in_seconds'))

     tmp = df_traks.groupby(['article_id']).agg([pl.col('weekday').median().alias('median_weekday_article_id'),
          pl.col('weekday').mean().alias('mean_weekday_article_id'),
          pl.col('hour').median().alias('median_hour_article_id'),
          pl.col('hour').mean().alias('mean_hour_article_id'),]
     )
     df = df.join(tmp, on=['article_id'], how='left')

     tmp = df_traks.groupby(['customer_id']).agg([pl.col('weekday').median().alias('median_weekday_customer_id'),
          pl.col('weekday').mean().alias('mean_weekday_customer_id'),
          pl.col('hour').median().alias('median_hour_customer_id'),
          pl.col('hour').mean().alias('mean_hour_customer_id'),]
     )
     df = df.join(tmp, on=['customer_id'], how='left')

     df = df.with_columns((pl.col('mean_hour_article_id') / pl.col('mean_hour_customer_id')).alias('mean_hour_diff'))

     #df = df[(df['rnk_tifuknn_tfidf']>=0) | (df['rnk_tifu_cos']>=0)]
     return df

In [42]:
df = make_features(df_train_traks.filter(pl.col('customer_id')>=0), mode='train'#, item_factors, user_factors
)
df.write_parquet('data/lb_features_train.par')

df = make_features(df_train_all.filter(pl.col('customer_id')>=0), mode='predict'#, item_factors, user_factors
)
df.write_parquet('data/df_lb_predict_all.par')

100%|██████████| 12/12 [00:21<00:00,  1.79s/it]


In [ ]:
0.814013

In [ ]:
8129

In [4]:
df = make_features(df_train_all.filter(pl.col('customer_id')>=0), mode='predict'#, item_factors, user_factors
)
df.write_parquet('data/df_lb_predict_all.par')

100%|██████████| 27/27 [01:01<00:00,  2.28s/it]


df_test = df_test_traks.to_pandas()
type_labels = {0: 'clicks', 1: 'carts', 2: 'orders'}
df_test['type'] = df_test['type'].map(type_labels)

joint_pred = df_train_traks_tmp.to_pandas().sort_values('rnk')#.rename({'article_id_right': 'article_id'}, axis=1)
submission = utils.sub_for_metrics(joint_pred, df_train_all.to_pandas(), list(df_test_traks['customer_id'].unique()))
utils.get_metric(submission, df_test)

In [146]:
def time_cat(df, days=7):
    return df.filter(pl.col('ts') >= df['ts'].max() - days*1000*60*60*24)

In [147]:
df_track = time_cat(df_train_traks, days=7)

In [39]:
def add_pop_items(df, df_traks, pred_users):
    df_track = time_cat(df_traks, days=7)
    top_pop_items = pd.DataFrame(df_track[df_track['type']!='clicks']['article_id'].value_counts()[:25].keys(), columns=['article_id'])
    df_users = pd.DataFrame(pred_users, columns=['customer_id'])
    count_rec = df['customer_id'].value_counts()
    top_pop_items['key'] = 0
    df_users['key'] = 0
    top_pop_items_min = top_pop_items.head(10).merge(df_users[df_users['customer_id'].isin(count_rec[count_rec>50].keys())], 
        on='key', how='outer').drop(columns=['key'])
    top_pop_items_max = top_pop_items.head(50).merge(df_users[df_users['customer_id'].isin(count_rec[count_rec>50].keys())], 
        on='key', how='outer').drop(columns=['key'])

    top_pop_items = pd.concat([top_pop_items_min, top_pop_items_max])
    df = pd.merge(df, top_pop_items, how='outer', on=['article_id', 'customer_id'])
    return df

In [ ]:
df_train = add_pop_items(df_train, df_train_traks, df_test_traks['customer_id'].unique())

In [16]:
def add_last(df, df_track, pred_users):
    df_track = df_track[df_track['customer_id'].isin(pred_users)]
    df = pd.merge(df, df_track[['customer_id', 'article_id']].drop_duplicates().groupby('customer_id').head(20), on=['customer_id', 'article_id'], how='outer')
    return df

In [17]:
print(df_train.shape)
df_train = add_last(df_train, df_train_traks, df_test_traks['customer_id'].unique())
print(df_train.shape)

(44271848, 10)
(44273991, 10)


In [5]:
123

123

In [13]:
neg_sample = len(df_train_candidat[df_train_candidat['target']==1])*20
df_train_candidat = df_train_candidat[df_train_candidat['target']==1].append(df_train_candidat[df_train_candidat['target']==0].sample(neg_sample, random_state=42))

In [14]:
df_train_candidat['target'] = df_train_candidat['target'].astype(np.int32)

In [15]:
df_train_candidat['target'].value_counts()#/l≈en(df_traian_candidat)

0    2589080
1     129454
Name: target, dtype: int64

In [19]:
df_train_candidat['target'].value_counts()#/l≈en(df_traian_candidat)

0    2593300
1     129665
Name: target, dtype: int64

In [11]:
df_train_candidat['target'].value_counts()#/l≈en(df_traian_candidat)

0    2587940
1     129397
Name: target, dtype: int64

In [15]:
df_train_candidat['target'].value_counts()#/l≈en(df_traian_candidat)

0    2553420
1     127671
Name: target, dtype: int64

In [12]:
df_train_candidat['target'].value_counts()#/l≈en(df_traian_candidat)

0    2498300
1     124915
Name: target, dtype: int64

In [25]:
df_train_candidat['target'].value_counts()#/l≈en(df_traian_candidat)

0    2526700
1     126335
Name: target, dtype: int64

In [6]:
def time_cat(df, days=7):
    validation_cut = df['t_dat'].max() - pd.Timedelta(days=days)
    return df[df['t_dat'] >= validation_cut]

def make_features(df, df_traks, mode='metric'#, item_factors, user_factors
):
     # расстояние между действием и покупкой
     # Среднее время между событиями
     # фичи по времени и дню недели как профиль клиента 
     df_traks['hour'] = df_traks['t_dat'].dt.hour
     df_traks['weekday'] = df_traks['t_dat'].dt.weekday
     
     # Средняя неделя и час покупки товара


     if mode=='train':
          df = df[['customer_id', 'article_id', 'target']]
     else:
          df = df[['customer_id', 'article_id']]
     
     last_view = (df_traks['t_dat'].max() - df_traks.groupby(['article_id'])['t_dat'].max() ).dt.total_seconds()/60
     df['last_view'] = df['article_id'].map(last_view).astype(np.float32)

     last_buy = (df_traks[df_traks['type']!='clicks']['t_dat'].max() - df_traks.groupby(['article_id'])['t_dat'].max() ).dt.total_seconds()/60
     df['last_buy'] = df['article_id'].map(last_buy).astype(np.float32)

     joint_traks = get_joint_listren(df_traks, 'article_id')
     df = agg_ratio_joint_sales(df, joint_traks, df_traks, pred_users=df['customer_id'].unique(), N=15)
     #return df, joint_traks
     df = agg_ratio_joint_sales(df, joint_traks, df_traks, pred_users=df['customer_id'].unique(), N=5)
     df = agg_ratio_joint_sales(df, joint_traks, df_traks, pred_users=df['customer_id'].unique(), N=2)
     del joint_traks
     df_orders = df_traks[df_traks['type']=='orders']
     df_orders = df_orders.sort_values('t_dat', ascending=False)
     df_orders['num_action'] = df_orders.groupby('customer_id')['article_id'].cumcount()
     joint_traks = get_joint_listren(df_orders, 'article_id')

     df = agg_ratio_joint_sales(df, joint_traks, df_orders, pred_users=df['customer_id'].unique(), N=6)

     df_tmp = df_traks.groupby(['customer_id', 'article_id'])['num_action'].min().to_frame('min_num_action').reset_index()
     df = pd.merge(df, df_tmp, on=['customer_id', 'article_id'], how='left')

     df_tmp = df_traks.groupby(['customer_id', 'article_id'])['t_dat'].count().to_frame('user_item_count').reset_index()
     df = pd.merge(df, df_tmp, on=['customer_id', 'article_id'], how='left')

     df_tmp = df_traks[df_traks['type']=='carts'].groupby(['customer_id', 'article_id'])['t_dat'].count().to_frame('user_item_carts_count').reset_index()
     df = pd.merge(df, df_tmp, on=['customer_id', 'article_id'], how='left')

     df_tmp = df_traks[df_traks['type']=='orders'].groupby(['customer_id', 'article_id'])['t_dat'].count().to_frame('user_item_orders_count').reset_index()
     df = pd.merge(df, df_tmp, on=['customer_id', 'article_id'], how='left')

     # df = pd.merge(df, joint_traks.rename({'article_id_x': 'article_id', 'article_id_y': 'last_article_id'}, axis=1), how='left', on=['article_id', 'last_article_id'])
     # df = df.drop(columns=['last_article_id'])
     df_implicit = pd.read_parquet(f'data/implicit_lb_{mode}.par') # predict
     df = pd.merge(df, df_implicit, on=['article_id', 'customer_id'], how='left')
     df_implicit = pd.read_parquet(f'data/implicit_lb_{mode}10.par') # predict
     df = pd.merge(df, df_implicit, on=['article_id', 'customer_id'], how='left')


     # df_implicit = pd.read_parquet(f'data/implicit_lb_{mode}_orders.par') # predict
     # df = pd.merge(df, df_implicit, on=['article_id', 'customer_id'], how='left')

     gc.collect()


     # Среднии, минимальные и максимальные ранги после модели первого уровня для артистов и треков

     df['mean_rnk_cand'] = df[['rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].mean(axis=1).astype(np.float32)
     df['mean_rnk_cand'] = df[['rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].std(axis=1).astype(np.float32)
     df['min_rnk_cand'] = df[['rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].min(axis=1).astype(np.float32)
     df['max_rnk_cand'] = df[['rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].max(axis=1).astype(np.float32)

     df['mean_rnk_cand_10'] = df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos','rnk_tfidf_10',  'rnk_cos412_10']].mean(axis=1).astype(np.float32)
     df['mean_rnk_cand_10'] = df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos','rnk_tfidf_10',  'rnk_cos412_10']].std(axis=1).astype(np.float32)
     df['min_rnk_cand_10'] = df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos','rnk_tfidf_10',  'rnk_cos412_10']].min(axis=1).astype(np.float32)
     df['max_rnk_cand_10'] = df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos','rnk_tfidf_10',  'rnk_cos412_10']].max(axis=1).astype(np.float32)

     df['mean_rnk_cand_all'] = df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos','rnk_tfidf_10',  'rnk_cos412_10',
                                   'rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].mean(axis=1).astype(np.float32)
     df['mean_rnk_cand_all'] = df[['rnk_tifuknn_tfidf_10', 'rnk_tifu_cos','rnk_tfidf_10',  'rnk_cos412_10',
                                   'rnk_tifuknn_tfidf', 'rnk_tifu_cos','rnk_tfidf',  'rnk_cos412', 'rnk_BM25']].std(axis=1).astype(np.float32)


     # фичи на популярность пользователя, товара 
     df['pop_article_id'] = df['article_id'].map(df_traks['article_id'].value_counts()).astype(np.int32)
     df['nunique_article_id'] = df['customer_id'].map(df_traks.groupby('customer_id')['article_id'].nunique()).astype(np.int32)
     df_traks['pop_article_id'] = df_traks['article_id'].map(df_traks['article_id'].value_counts()).astype(np.int32)

     df['pop_user'] = df['customer_id'].map(df_traks['customer_id'].value_counts()).astype(np.int32)
     df['pop_user_orders'] = df['customer_id'].map(df_traks[df_traks['type']=='orders']['customer_id'].value_counts()).astype(np.float32)
     df['pop_user_carts'] = df['customer_id'].map(df_traks[df_traks['type']=='carts']['customer_id'].value_counts()).astype(np.float32)
     # Ранги песен внутри кандидатов
     df['pop_item_rnk_user'] = df.groupby('customer_id')['pop_article_id'].rank(method='dense', ascending=False).astype(np.float32)
     # Какие треки в среднем по популярности слушает пользователь
     for col in ['article_id']:
          df_traks[f'pop_{col}'] = df_traks['article_id'].map(df_traks.groupby([col])['customer_id'].count()).astype(np.float32)
          df[f'pop_{col}_mean_user'] = df['customer_id'].map(df_traks.groupby(['customer_id'])[f'pop_{col}'].mean()).astype(np.float32)
          df[f'pop_{col}_std_user'] = df['customer_id'].map(df_traks.groupby(['customer_id'])[f'pop_{col}'].std()).astype(np.float32)
          #df[f'pop_{col}_min_user'] = df['customer_id'].map(df_traks.groupby(['customer_id'])[f'pop_{col}'].min()).astype(np.float32)
          #df[f'pop_{col}_max_user'] = df['customer_id'].map(df_traks.groupby(['customer_id'])[f'pop_{col}'].max()).astype(np.float32)
          df[f'ratio_pop_{col}_user_item'] = (df[f'pop_{col}_mean_user'] / df[f'pop_{col}']).astype(np.float32)

     for i in [5, 10, 15]:
          tmp_track = time_cat(df_traks, days=i)
         # tmp_track = df_traks[df_traks['num_action']<=i]
          df[f'pop_item_{i}'] = df['article_id'].map(tmp_track.groupby(['article_id'])['customer_id'].count()).astype(np.float32)
          df[f'pop_item_orders_{i}'] = df['article_id'].map(tmp_track[tmp_track['type']=='orders'].groupby(['article_id'])['customer_id'].count()).astype(np.float32)
          gc.collect()
     df['min_date_session'] = df['customer_id'].map(df_traks.groupby('customer_id')['t_dat'].min())
     df['max_date_session'] = df['customer_id'].map(df_traks.groupby('customer_id')['t_dat'].max())
     df['leng_session'] =( df['max_date_session'] - df['min_date_session'] ).dt.total_seconds()
     df['items_in_seconds'] = df['pop_user'] / df['leng_session']
     df = df.drop(columns=['max_date_session', 'min_date_session'])

     df[f'mode_weekday_article_id'] = df['article_id'].map(df_traks.groupby(['article_id'])['weekday'].median()).astype(np.float32)
     df[f'mean_weekday_article_id'] = df['article_id'].map(df_traks.groupby(['article_id'])['weekday'].mean()).astype(np.float32)
     df[f'mode_hour_article_id'] = df['article_id'].map(df_traks.groupby(['article_id'])['hour'].median()).astype(np.float32)
     df[f'mean_hour_article_id'] = df['article_id'].map(df_traks.groupby(['article_id'])['hour'].mean()).astype(np.float32)
     # Средняя неделя и час покупки клиента
     df[f'mode_weekday_customer_id'] = df['customer_id'].map(df_traks.groupby(['customer_id'])['weekday'].median()).astype(np.float32)
     df[f'mean_weekday_customer_id'] = df['customer_id'].map(df_traks.groupby(['customer_id'])['weekday'].mean()).astype(np.float32)
     df[f'mode_hour_customer_id'] = df['customer_id'].map(df_traks.groupby(['customer_id'])['hour'].median()).astype(np.float32)
     df[f'mean_hour_customer_id'] = df['customer_id'].map(df_traks.groupby(['customer_id'])['hour'].mean()).astype(np.float32)

     df[f'mean_hour_diff'] = (df[f'mean_hour_article_id'] - df[f'mean_hour_customer_id']).astype(np.float32)

     #df = df[(df['rnk_tifuknn_tfidf']>=0) | (df['rnk_tifu_cos']>=0)]
     return df

In [19]:
df_train = make_features(df_train_candidat, df_train_traks[df_train_traks['customer_id']>=0], mode='train'#, item_factors, user_factors
)

end agg_ratio_joint_sales
end agg_ratio_joint_sales
end agg_ratio_joint_sales
end agg_ratio_joint_sales


In [23]:
# Добавить последнего артиста и фичу на равенство артиста
# Усреднить предсказания моделей первого уровня
# сколько треков назад лайкал трек этого артиста

In [ ]:
# Добавить ранги популярности треков и артивтов внутри пользователей

df_test_traks['target'] = 1
df_train = pd.merge(df_train, df_test_traks[df_test_traks['type']!='clicks'][['target', 'customer_id', 'article_id']],
                             how='left', on=['customer_id', 'article_id'])
df_train['target'] = df_train['target'].fillna(0)

In [21]:
df_train.to_parquet('data/lb_features_train.par')

In [17]:
df_train = df_train.drop(columns=['nunique_article_id'])

In [26]:
df_train = df_train.drop(columns=['item_date_last'])#.head()

In [17]:
df_train = pd.read_parquet('data/lb_features_train.par')

In [16]:
df_train['target'].value_counts()#/len(df_train)

0    2611600
1     130580
Name: target, dtype: int64

In [30]:
df_train['target'].value_counts()#/len(df_train)

0.0    41897780
1.0     3427921
Name: target, dtype: int64

In [17]:
import gc
gc.collect()

17

In [22]:
if PREDICT_LB:
    df_predicted = implicit_features.make_pred(df_train_all, test_users, filter_already_liked_items=False)
else:
    df_predicted = implicit_features.make_pred(df_train_transactions_all, df_val_transactions['customer_id'].unique())

  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [05:04<00:00, 5487.54it/s]


  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [03:37<00:00, 7684.13it/s]


  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [05:16<00:00, 5277.39it/s]


  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [05:31<00:00, 5036.86it/s]


  0%|          | 0/1855542 [00:00<?, ?it/s]

100%|██████████| 1671802/1671802 [04:51<00:00, 5733.56it/s]


In [43]:
df_predicted_orders_carts = implicit_features.make_pred(df_train_all[df_train_all['type']!='clicks'], 
                                    test_users, filter_already_liked_items=False)
                                    
for col in (set(df_predicted_orders_carts.columns) - set(['article_id', 'customer_id'])):
    df_predicted_orders_carts = df_predicted_orders_carts.rename({col: col+'_order'}, axis=1)

  0%|          | 0/475424 [00:00<?, ?it/s]

100%|██████████| 229688/229688 [00:27<00:00, 8415.12it/s]


  0%|          | 0/475424 [00:00<?, ?it/s]

100%|██████████| 229688/229688 [00:27<00:00, 8346.93it/s]


  0%|          | 0/475424 [00:00<?, ?it/s]

100%|██████████| 229688/229688 [00:27<00:00, 8259.54it/s]


  0%|          | 0/475424 [00:00<?, ?it/s]

100%|██████████| 229688/229688 [00:24<00:00, 9255.10it/s]


In [44]:
df_predicted_orders_carts.to_parquet('data/implicit_lb_predict_orders.par')

In [23]:
df_predicted_10 = implicit_features.make_pred(df_train_all, 
                                      test_users, train_day=10)

  0%|          | 0/1619946 [00:00<?, ?it/s]

100%|██████████| 1668512/1668512 [05:09<00:00, 5398.40it/s]


  0%|          | 0/1619946 [00:00<?, ?it/s]

100%|██████████| 1668512/1668512 [03:30<00:00, 7936.27it/s]


  0%|          | 0/1619946 [00:00<?, ?it/s]

100%|██████████| 1668512/1668512 [05:03<00:00, 5495.41it/s]


  0%|          | 0/1619946 [00:00<?, ?it/s]

100%|██████████| 1668512/1668512 [05:21<00:00, 5194.33it/s]


  0%|          | 0/1619946 [00:00<?, ?it/s]

100%|██████████| 1668512/1668512 [04:59<00:00, 5566.70it/s]


In [ ]:
for col in set(df_predicted_10.columns) - set(['article_id', 'customer_id']):
    df_predicted_10 = df_predicted_10.rename(columns={col: col+'_10'})


In [ ]:
df_predicted_10.to_parquet('data/implicit_lb_predict10.par')

In [ ]:
if PREDICT_LB:
    df_predicted.to_parquet('data/implicit_lb_predict.par')

In [7]:
df_predicted = pd.read_parquet('data/implicit_lb_predict.par', columns=['customer_id', 'article_id'])
#df_predicted = pd.merge(df_predicted, pd.read_parquet('data/implicit_lb_predict_orders.par'), on=['article_id', 'customer_id'], how='outer')

In [8]:
df_predicted_10 = pd.read_parquet('data/implicit_lb_predict10.par', columns=['customer_id', 'article_id'])

In [9]:
df_predicted = pd.merge(df_predicted, df_predicted_10, on=['article_id', 'customer_id'], how='outer')

In [7]:
import public_candidates
public_candidates_predict = public_candidates.predict(df_train_all, 
                                                    df_test)

  0%|          | 0/14231 [00:00<?, ?it/s]

  0%|          | 0/14231 [00:02<?, ?it/s]

  0%|          | 0/1633 [00:01<?, ?it/s]

In [8]:
public_candidates_predict.to_parquet('data/public_cand_lb_predict.par')

In [7]:
public_candidates_predict = pd.read_parquet('data/public_cand_lb_predict.par', columns=['customer_id', 'article_id'])

In [8]:
df_predicted = pd.merge(df_predicted, public_candidates_predict, on=['article_id', 'customer_id'], how='outer')

In [9]:
del public_candidates_predict, df_predicted_10

In [ ]:
df_predicted = add_pop_items(df_predicted, df_train_all, test_users)

In [12]:
joint_traks = get_joint_listren(df_train_all, 'article_id')
df_predicted = agg_ratio_joint_sales(df_predicted, joint_traks, df_train_all, test_users, N=15, for_candidate=True )

In [20]:
df_predicted = agg_ratio_joint_sales(df_predicted, joint_traks, df_train_all, test_users, N=15, for_candidate=True )

In [24]:
len(set(test_users) - set(df_predicted['customer_id']))

49

In [21]:
df_predicted.shape

(156713703, 2)

In [48]:
df_predicted_10 = pd.read_parquet('data/predicted_10_boosting.par', columns=['rnk_tifuknn_tfidf', 'rnk_BM25', 'rnk_tifu_cos', 
    'rnk_tfidf', 'rnk_cos412', 'customer_id', 'article_id'])

df_test_als = implicit_features.als(df_train_all, test_users)

df_test_als.to_parquet('data/df_test_als.par')

df_test_als = pd.read_parquet('data/df_test_als.par')

df_predicte_5 = pd.read_parquet('data/predicted_5_boosting.par')

df_predicte_5 = pd.read_parquet('data/predicted_5_boosting.par', columns=['rnk_tifuknn_tfidf_5', 'rnk_BM25_5', 'rnk_tifu_cos_5', 
    'rnk_tfidf_5', 'rnk_cos412_5', 'customer_id', 'article_id'])
df_predicted = pd.merge(df_predicted, df_predicte_5, on=['customer_id', 'article_id'], how='outer')

In [51]:
ratio_recall = pd.read_parquet('data/ratio_recall.par')

In [52]:
columns = set(df_predicted.columns) - set(['customer_id', 'article_id', 'target'])
result_col = []
rnk_col = []
for col in tqdm(columns):
    if 'rnk' in col:
        result_col.append(df_predicted[col].isna().values)
        rnk_col.append((df_predicted[col]<=ratio_recall.loc[col][0]).values)
        gc.collect()

drop_row = np.sum(np.array(result_col), axis=0)
drop_row_rnk = np.sum(np.array(rnk_col), axis=0)
#del result_col, rnk_col
drop_index = df_predicted[(drop_row==max(drop_row)) & (drop_row_rnk==0)].index
print(len(drop_index))
df_predicted.drop(drop_index, inplace=True)

100%|██████████| 10/10 [00:02<00:00,  3.88it/s]


15324735


df_train_als = pd.read_parquet('data/df_test_als.par')
df_train_als = df_train_als.sort_values('score_als', ascending=False)
df_train_als['rnk_als'] = df_train_als.groupby('customer_id')['score_als'].cumcount()
df_train_als = df_train_als[df_train_als['rnk_als']<70]
df_predicted = pd.merge(df_predicted, df_train_als, on=['customer_id', 'article_id'], how='outer')

df_train_elsa = pd.read_parquet('data/df_predicted_predict_elsa.par')
df_train_elsa = df_train_elsa.sort_values('pred', ascending=False)
df_train_elsa['rnk_elsa'] = df_train_elsa.groupby('customer_id')['pred'].cumcount()
df_train_elsa = df_train_elsa[df_train_elsa['rnk_elsa']<85]
df_predicted = pd.merge(df_predicted, df_train_elsa[['customer_id', 'article_id', 'pred']], on=['customer_id', 'article_id'], how='outer')

df_predicted = pd.merge(df_predicted, df_test_als, on=['customer_id', 'article_id'], how='outer')

In [53]:
pop_items = pop_candidates.get_pop_candidates(df_train_all, test_users)
df_predicted = pd.merge(df_predicted, pop_items, on=['customer_id', 'article_id'], how='outer')

In [54]:
pop_artist_user = pop_candidates.get_pop_artist_items(df_train_all, test_users, df_test_artist)
df_predicted = pd.merge(df_predicted, pop_artist_user[['customer_id', 'article_id']], on=['customer_id', 'article_id'], how='outer')

In [10]:
df_predicted.head()

,customer_id,article_id
0,12899779,827628
1,12899779,1654515
2,12899779,1019683
3,12899779,1606540
4,12899779,1559470


In [11]:
result = []
for indx in tqdm(np.array_split(df_predicted['customer_id'].unique(), 10)):
    result.append(make_features(df_predicted[df_predicted['customer_id'].isin(indx)], df_train_all, mode = 'predict'))


  0%|          | 0/10 [00:00<?, ?it/s]

In [31]:
df_predicted.to_parquet('data/df_lb_predict_all.par')

In [5]:
df_predicted = pd.read_parquet('data/df_lb_predict_all.par')

In [6]:
import gc
gc.collect()

0

In [14]:
df_train_all[df_train_all['customer_id']>=0].head()

,customer_id,article_id,ts,type,t_dat,num_action
434719298,14571581,1100210,1662328791563,clicks,2022-09-04 21:59:51.563,0
434719218,14571515,781377,1662328789887,clicks,2022-09-04 21:59:49.887,0
434718583,14571170,336732,1662328788088,clicks,2022-09-04 21:59:48.088,0
434710696,14568399,104775,1662328781349,clicks,2022-09-04 21:59:41.349,0
434719297,14571580,202353,1662328781067,clicks,2022-09-04 21:59:41.067,0


In [10]:
df_predicted = make_features(df_predicted, df_train_all[df_train_all['customer_id']>=0], mode = 'predict'#, item_factors, user_factors
)
df_predicted.to_parquet('data/df_lb_predict_all.par')

end agg_ratio_joint_sales
end agg_ratio_joint_sales
end agg_ratio_joint_sales
end agg_ratio_joint_sales


In [11]:
df_predicted.to_parquet('data/df_lb_predict_all.par')

In [12]:
df_predicted.drop_duplicates(subset=['customer_id', 'article_id']).shape

(223537340, 69)

In [13]:
df_predicted.shape

(223537340, 69)

In [30]:
df_predicted = pd.read_parquet('data/df_predicted_all_lgbm.par')

In [31]:
df_predicted = df_predicted.sort_values('pred', ascending=False)

In [32]:
utils.makesub(df_predicted, df_test, name='add_features')

In [53]:
df_predicted.loc[int(len(df_predicted)//2):, 'pred'] = model.predict_proba(df_predicted.loc[int(len(df_predicted)//2):, features])[:,1]
df_predicted.loc[:int(len(df_predicted)//2), 'pred'] = model.predict_proba(df_predicted.loc[:int(len(df_predicted)//2), features])[:,1]

AttributeError: 'BoostingModel' object has no attribute 'predict_proba'

In [57]:
df_predicted['pred'] = model.predict_lgbm(models, df_predicted, features)

next_mode;
next_mode;
next_mode;
next_mode;
next_mode;


In [26]:
df_orders = df[df['type']=='orders'][['customer_id', 'article_id', 'num_action']]
df_orders = df_orders.drop_duplicates(subset=['customer_id', 'article_id'])

In [28]:
df_predicted = pd.merge(df_predicted, df_orders, how='left', on=['customer_id', 'article_id'])

In [ ]:
df_predicted = df_predicted[df_predicted['num_action'].isna()]

In [ ]:
df_predicted = df_predicted.sort_values('pred', ascending=False)

In [ ]:
utils.makesub(df_predicted, df_test, name='submission_lgm_orders_drop')

In [118]:
df_predicted.to_parquet('data/df_predicted_all_lgbm.par')

In [77]:
# Выбирать по реколу каждого кандидата